# Chapter 16 — Measures

Companion notebook to Chapter 16 of *Advanced Calculus for Machine Learning*.
Every numerical claim of the chapter is recomputed here with PyTorch and
checked with an assertion.

Measures of sets of the plane are computed by counting the points of a fine
regular grid, which is deterministic. Random samples are used only in the
last section, where a cell says so.

In [1]:
import math
import itertools
import warnings
import torch

warnings.filterwarnings("ignore", category=FutureWarning)
torch.set_default_dtype(torch.float64)
torch.manual_seed(0)
print(torch.__version__)
T = torch.tensor

def grid_area(inside, lo, hi, n=1500):
    # Lebesgue measure of {x in [lo, hi)^2 : inside(x)}, by counting the centres of n^2 cells
    h = (hi - lo) / n
    c = lo + h * (torch.arange(n) + 0.5)
    X, Y = torch.meshgrid(c, c, indexing="ij")
    return inside(torch.stack([X, Y], -1)).sum().item() * h * h

2.15.0+cu132


## 1. Why a new integral

Section 16.1: the indicator of the first `n` rationals has Riemann integral
0, but for the limit function the Riemann sums depend on the points where the
function is evaluated.

In [2]:
from fractions import Fraction
def rationals(max_den):                         # the rationals of [0, 1] with denominators up to max_den
    return sorted({Fraction(p, q) for q in range(1, max_den + 1) for p in range(q + 1)})
Q = rationals(40)
print("rationals of [0, 1] with denominator at most 40:", len(Q))
# Riemann sums of the indicator of these finitely many points: at most (number of points) x (mesh)
for N in (10 ** 3, 10 ** 5):
    upper = min(len(Q), N) * 2 / N              # each point touches at most two intervals of length 1/N
    print(f"subdivision into {N} intervals: every Riemann sum is at most {upper:.4f}")
assert min(len(Q), 10 ** 5) * 2 / 10 ** 5 < 0.02
# the limit function f = 1 on the rationals: tags k/N are rational, tags (k + 1/sqrt(2))/N are not
N = 1000
rational_tags = [Fraction(k, N) for k in range(N)]
sum_rational = sum(1 for t in rational_tags) / N        # f = 1 at every rational tag
sum_irrational = 0.0                                    # f = 0 at every irrational tag
print("Riemann sums of the limit function: with rational tags", sum_rational, ", with irrational tags", sum_irrational)
assert sum_rational == 1.0 and sum_irrational == 0.0

rationals of [0, 1] with denominator at most 40: 491
subdivision into 1000 intervals: every Riemann sum is at most 0.9820
subdivision into 100000 intervals: every Riemann sum is at most 0.0098
Riemann sums of the limit function: with rational tags 1.0 , with irrational tags 0.0


## 2. σ-algebras on finite sets

Example 16.5, Exercise 16.1 and Exercise 16.4: the σ-algebra generated by a
family of subsets is computed by closing it under complements and unions.

In [3]:
def generate(X, family):
    X = frozenset(X); sets = {frozenset(), X} | {frozenset(s) for s in family}
    while True:
        new = {X - a for a in sets} | {a | b for a in sets for b in sets}
        if new <= sets:
            return sets
        sets |= new
X6 = range(1, 7)
S = generate(X6, [{1, 2}, {3}, {4, 5, 6}])
print("Exercise 16.1:", sorted(sorted(s) for s in S))
assert len(S) == 8
def measurable(f, sigma, X):
    return all(frozenset(x for x in X if f(x) == v) in sigma for v in {f(x) for x in X})
print("x -> x:", measurable(lambda x: x, S, X6), "  1_{x <= 2}:", measurable(lambda x: x <= 2, S, X6),
      "  min(x, 4):", measurable(lambda x: min(x, 4), S, X6))
assert not measurable(lambda x: x, S, X6) and measurable(lambda x: x <= 2, S, X6) and not measurable(lambda x: min(x, 4), S, X6)
# a partition into m blocks generates 2^m sets
assert len(generate(range(12), [{0, 1, 2}, {3}, {4, 5}, {6, 7, 8, 9}, {10, 11}])) == 2 ** 5

# Exercise 16.4
X4 = frozenset({1, 2, 3, 4})
subsets = [frozenset(c) for r in range(5) for c in itertools.combinations(X4, r)]
even = {s for s in subsets if len(s) % 2 == 0}
is_lambda = X4 in even and all((b - a) in even for a in even for b in even if a <= b)   # increasing unions: automatic on a finite set
is_pi = all((a & b) in even for a in even for b in even)
print("even subsets: lambda-system", is_lambda, "  pi-system", is_pi)
assert is_lambda and not is_pi
C = [{1, 2}, {2, 3}]
assert len(generate(X4, C)) == 16                                # sigma(C) is the family of all subsets
mu = {1: 0.25, 2: 0.25, 3: 0.25, 4: 0.25}; nu = {1: 0.5, 2: 0.0, 3: 0.5, 4: 0.0}
m = lambda w, s: sum(w[x] for x in s)
assert all(m(mu, s) == m(nu, s) for s in C) and m(mu, {2}) != m(nu, {2})
agree = {s for s in subsets if m(mu, s) == m(nu, s)}
print("sets on which the two probabilities agree:", sorted(sorted(s) for s in agree))
assert X4 in agree and all((b - a) in agree for a in agree for b in agree if a <= b)    # a lambda-system
assert not all((a & b) in agree for a in agree for b in agree)                          # not a pi-system

Exercise 16.1: [[], [1, 2], [1, 2, 3], [1, 2, 3, 4, 5, 6], [1, 2, 4, 5, 6], [3], [3, 4, 5, 6], [4, 5, 6]]
x -> x: False   1_{x <= 2}: True   min(x, 4): False
even subsets: lambda-system True   pi-system False
sets on which the two probabilities agree: [[], [1, 2], [1, 2, 3, 4], [1, 4], [2, 3], [3, 4]]


## 3. Measures

Proposition 16.8 and Theorem 16.9 on weighted counting measures, and the
pitfall on decreasing sequences.

In [4]:
torch.manual_seed(1)
w = torch.rand(30)                                              # a measure on {0, ..., 29}: mu(A) = sum of the weights
mu_ = lambda A: w[sorted(A)].sum().item() if A else 0.0
A = set(range(0, 18)); B = set(range(10, 25))
assert abs(mu_(A | B) + mu_(A & B) - mu_(A) - mu_(B)) < 1e-12
assert mu_(A & B) <= mu_(A) and abs(mu_(A - B) - (mu_(A) - mu_(A & B))) < 1e-12
covers = [set(range(i, i + 7)) for i in range(0, 24, 3)]
assert mu_(set().union(*covers)) <= sum(mu_(c) for c in covers) + 1e-12
# continuity from below on an infinite set: mu = sum_k 2^-k delta_k, A_n = {1, ..., n}
print("mu({1,...,n}) for n = 5, 10, 20, 50:", [1 - 2.0 ** -n for n in (5, 10, 20, 50)], " -> mu(all) = 1")
# from above without finiteness: counting measure of {n, n+1, ...} is infinite for every n; the intersection is empty
tail = lambda n, N: N - n + 1                                   # number of elements of {n, ..., N}
print("counting measure of {n, ..., N} for n = 10 and N = 10^3, 10^6, 10^9:", [tail(10, 10 ** k) for k in (3, 6, 9)])
# Lebesgue: (0, 1/n) decreases to the empty set and its length tends to 0; [n, infinity) has infinite length
print("lengths of (0, 1/n):", [1 / n for n in (1, 10, 100, 1000)])
# the rationals with denominator <= 40 covered by intervals of total length at most eps
eps = 1e-3
total = sum(eps * 2.0 ** -(k + 1) for k in range(len(Q)))
print("total length of the cover of", len(Q), "rationals:", f"{total:.6f}", "<=", eps)
assert total <= eps

mu({1,...,n}) for n = 5, 10, 20, 50: [0.96875, 0.9990234375, 0.9999990463256836, 0.9999999999999991]  -> mu(all) = 1
counting measure of {n, ..., N} for n = 10 and N = 10^3, 10^6, 10^9: [991, 999991, 999999991]
lengths of (0, 1/n): [1.0, 0.1, 0.01, 0.001]
total length of the cover of 491 rationals: 0.001000 <= 0.001


## 4. Push-forwards

Definition 16.16, Example 16.17 and Exercise 16.5.

In [5]:
def pushforward(weights, f):
    outw = {}
    for x, p in weights.items():
        outw[f(x)] = outw.get(f(x), 0) + p
    return outw
die = {x: Fraction(1, 6) for x in range(1, 7)}
pf1 = pushforward(die, lambda x: x % 3); pf2 = pushforward(die, lambda x: int(x >= 5))
print("x mod 3:", {k: str(v) for k, v in sorted(pf1.items())}, "   1_{x >= 5}:", {k: str(v) for k, v in sorted(pf2.items())})
assert pf1 == {0: Fraction(1, 3), 1: Fraction(1, 3), 2: Fraction(1, 3)} and pf2 == {0: Fraction(2, 3), 1: Fraction(1, 3)}
assert sum(pf1.values()) == 1                                    # the total mass is preserved
g = lambda y: y * y
assert pushforward(pushforward(die, lambda x: x % 3), g) == pushforward(die, lambda x: g(x % 3))   # (g o f)_# = g_# f_#
assert pushforward({7: 1.0}, lambda x: x + 1) == {8: 1.0}        # the push-forward of a Dirac measure
assert pushforward(die, lambda x: 0) == {0: 1}                   # a constant map concentrates everything on one point

# the doubling map preserves Lebesgue measure on [0, 1): the preimage of [a, b) has length b - a
n = 2_000_000
u = (torch.arange(n) + 0.5) / n
fu = (2 * u) % 1.0
for a, b in ((0.0, 0.3), (0.25, 0.26), (0.5, 0.95)):
    frac = ((fu >= a) & (fu < b)).double().mean().item()
    pre = ((u >= a / 2) & (u < b / 2)) | ((u >= (a + 1) / 2) & (u < (b + 1) / 2))
    assert torch.equal(pre, (fu >= a) & (fu < b)) and abs(frac - (b - a)) < 1e-5
print("doubling map: f^-1([a, b)) = [a/2, b/2) u [(a+1)/2, (b+1)/2), of length b - a")

x mod 3: {0: '1/3', 1: '1/3', 2: '1/3'}    1_{x >= 5}: {0: '2/3', 1: '1/3'}
doubling map: f^-1([a, b)) = [a/2, b/2) u [(a+1)/2, (b+1)/2), of length b - a


## 5. Lebesgue measure: translations and linear maps

Theorems 16.20 and 16.21, Figure 16.3, Proposition 16.23 and
Exercises 16.6 and 16.8.

In [6]:
disc = lambda p: (p ** 2).sum(-1) < 1
area_disc = grid_area(disc, -1.5, 1.5)
print("area of the unit disc on the grid:", round(area_disc, 5), "  pi =", round(math.pi, 5))
assert abs(area_disc - math.pi) < 2e-3
v = T([0.37, -0.21])
assert abs(grid_area(lambda p: disc(p - v), -1.5, 1.5) - area_disc) < 2e-3          # translation
# the linear map of Exercise 16.6
Tm = T([[2.0, 1.0], [0.0, 3.0]]); Ti = torch.linalg.inv(Tm)
area_img = grid_area(lambda p: disc(p @ Ti.T), -4.0, 4.0)                           # x in T(disc) iff T^-1 x in disc
print("area of T(disc):", round(area_img, 4), "  |det T| pi =", round(6 * math.pi, 4))
assert abs(area_img - 6 * math.pi) < 2e-2
a_, b_ = 1.7, 0.6
assert abs(grid_area(lambda p: (p[..., 0] / a_) ** 2 + (p[..., 1] / b_) ** 2 < 1, -2.0, 2.0) - math.pi * a_ * b_) < 5e-3
# T_# lambda = lambda / |det T|: the preimage of a set has its area divided by 6
square = lambda p: (p[..., 0] >= 0) & (p[..., 0] < 1) & (p[..., 1] >= 0) & (p[..., 1] < 1)
assert abs(grid_area(lambda p: square(p @ Tm.T), -1.0, 1.0) - 1 / 6) < 1e-3

# Figure 16.3: T = U Sigma V^T with rotations by 20 and 30 degrees and singular values 2, 0.75
rot = lambda deg: T([[math.cos(math.radians(deg)), -math.sin(math.radians(deg))], [math.sin(math.radians(deg)), math.cos(math.radians(deg))]])
U, Sg, V = rot(20), torch.diag(T([2.0, 0.75])), rot(30)
Tf = U @ Sg @ V.T
sq = T([[0.0, 0.0], [1.0, 0.0], [1.0, 1.0], [0.0, 1.0]])
def shoelace(P):
    x, y = P[:, 0], P[:, 1]
    return 0.5 * abs((x * torch.roll(y, -1) - y * torch.roll(x, -1)).sum().item())
stages = [sq, sq @ V, sq @ V @ Sg, sq @ Tf.T]                    # rows are points: p -> V^T p, Sigma V^T p, T p
print("areas of the four polygons:", [round(shoelace(P), 6) for P in stages])
assert [round(shoelace(P), 9) for P in stages] == [1.0, 1.0, 1.5, 1.5]
assert abs(torch.det(Tf).item() - 1.5) < 1e-12 and torch.allclose(torch.linalg.svdvals(Tf), T([2.0, 0.75]))
assert torch.allclose(stages[3], T([[0.0, 0.0], [1.756, 0.240], [2.473, 1.193], [0.717, 0.953]]), atol=1e-3)
# c(ST) = c(S) c(T), and a singular map sends everything into a line
S2 = T([[0.5, -1.0], [2.0, 1.0]])
assert abs(shoelace(sq @ (S2 @ Tm).T) - shoelace(sq @ S2.T) * shoelace(sq @ Tm.T)) < 1e-12
sing = T([[1.0, 2.0], [2.0, 4.0]])
assert shoelace(sq @ sing.T) < 1e-12

# Proposition 16.23 and Exercise 16.8: a frame with basis vectors of lengths 2 and 3 at 60 degrees
e1, e2 = T([2.0, 0.0]), 3 * T([math.cos(math.pi / 3), math.sin(math.pi / 3)])
G = T([[e1 @ e1, e1 @ e2], [e2 @ e1, e2 @ e2]])
cell = torch.stack([torch.zeros(2), e1, e1 + e2, e2])            # the unit square of the coordinates
print("Gram matrix:", [[round(v, 6) for v in row] for row in G.tolist()], "  sqrt(det G) =", round(math.sqrt(torch.det(G).item()), 6),
      "  area of the coordinate cell:", round(shoelace(cell), 6))
assert torch.allclose(G, T([[4.0, 3.0], [3.0, 9.0]])) and abs(math.sqrt(torch.det(G).item()) - 3 * math.sqrt(3)) < 1e-12
assert abs(shoelace(cell) - 3 * math.sqrt(3)) < 1e-12

area of the unit disc on the grid: 3.14182   pi = 3.14159
area of T(disc): 18.8494   |det T| pi = 18.8496
areas of the four polygons: [1.0, 1.0, 1.5, 1.5]
Gram matrix: [[4.0, 3.0], [3.0, 9.0]]   sqrt(det G) = 5.196152   area of the coordinate cell: 5.196152


## 6. Laws of random variables

Example 16.26 and Exercise 16.7: laws as push-forwards of the uniform
measure on `[0, 1)`, computed on a fine grid of `omega`.

In [7]:
n = 2_000_000
omega = (torch.arange(n) + 0.5) / n                               # a grid of Omega = [0, 1)
Z = -torch.log1p(-omega)
for t in (0.1, 1.0, 3.0):
    assert abs((Z <= t).double().mean().item() - (1 - math.exp(-t))) < 1e-6
print("Z = -ln(1 - U):  P(Z <= t) = 1 - exp(-t)")
die = torch.floor(6 * omega) + 1
assert all(abs((die == k).double().mean().item() - 1 / 6) < 1e-6 for k in range(1, 7))
for t in (0.2, 0.5, 0.9):
    assert abs((torch.sqrt(omega) <= t).double().mean().item() - t * t) < 1e-6
print("floor(6U) + 1 is uniform on {1, ..., 6};  P(sqrt(U) <= t) = t^2")

Z = -ln(1 - U):  P(Z <= t) = 1 - exp(-t)
floor(6U) + 1 is uniform on {1, ..., 6};  P(sqrt(U) <= t) = t^2


## 7. Machine learning: outputs and generative models

The two boxes of Section 16.8, Figure 16.4 and Exercise 16.9. This section
uses random samples.

In [8]:
torch.manual_seed(2)
net = torch.nn.Sequential(torch.nn.Linear(2, 8), torch.nn.Tanh(), torch.nn.Linear(8, 1)).double()
P_data = torch.randn(4000, 2)                                     # a sample: its empirical measure
Q_data = torch.randn(4000, 2) * 0.5 + T([1.5, -1.0])              # a shifted data distribution
with torch.no_grad():
    outP, outQ = net(P_data).squeeze(1), net(Q_data).squeeze(1)
# the push-forward of the empirical measure is the empirical measure of the outputs
B = (-0.2, 0.1)
in_B = lambda o: ((o > B[0]) & (o <= B[1])).double().mean().item()
print("mass given to (-0.2, 0.1] by the law of the output:  under P:", round(in_B(outP), 4), "  under Q:", round(in_B(outQ), 4))
assert abs(in_B(outP) - in_B(outQ)) > 0.05                         # the shift is visible on the outputs

# a generative model with a latent space of dimension 1: all samples on a parabola
z = torch.randn(200000)
samples = torch.stack([z, z ** 2], 1)
assert (samples[:, 1] - samples[:, 0] ** 2).abs().max() == 0       # mass 1 on the curve
for bins in (50, 200, 800):
    lo, hi = T([-3.0, 0.0]), T([3.0, 9.0])
    idx = ((samples - lo) / (hi - lo) * bins).long()
    ok = ((idx >= 0) & (idx < bins)).all(1)
    occupied = torch.unique(idx[ok], dim=0).shape[0]
    area = occupied * (6 / bins) * (9 / bins)
    print(f"{bins} x {bins} bins: {occupied} occupied, total area {area:.4f}")
assert area < 0.2
# Exercise 16.9: N rectangles of total area 8 k^3 / N cover the parabola above [-k, k]
k = 3.0
for N in (10, 100, 1000):
    h = 2 * k / N
    edges = -k + h * torch.arange(N + 1)
    heights = (edges[1:] ** 2 - edges[:-1] ** 2).abs()
    assert (heights <= 2 * k * h + 1e-12).all()
    print(f"N = {N}: area of the cover {(h * heights).sum().item():.4f}  <=  8 k^3 / N = {8 * k ** 3 / N:.4f}")
    assert (h * heights).sum().item() <= 8 * k ** 3 / N

mass given to (-0.2, 0.1] by the law of the output:  under P: 0.6697   under Q: 0.476


50 x 50 bins: 140 occupied, total area 3.0240


200 x 200 bins: 575 occupied, total area 0.7762
800 x 800 bins: 2244 occupied, total area 0.1893
N = 10: area of the cover 10.8000  <=  8 k^3 / N = 21.6000
N = 100: area of the cover 1.0800  <=  8 k^3 / N = 2.1600
N = 1000: area of the cover 0.1080  <=  8 k^3 / N = 0.2160


## Summary

* On a finite set a σ-algebra is the family of unions of the blocks of a
  partition; the sets on which two probabilities agree form a λ-system,
  which need not be stable by intersection.
* A push-forward preserves the total mass and composes like maps; the
  doubling map preserves Lebesgue measure.
* Lebesgue measure is invariant under translations and multiplied by
  `|det T|` under a linear map; orthogonal maps preserve it and the singular
  values account for the factor.
* Laws are push-forwards of the uniform measure; a generative model with a
  one-dimensional latent space puts all its mass on a set of area zero.